# Oxford-IIIT Pet: Inference and Augmentation

This notebook loads the multitask checkpoint produced by the training notebooks. It returns:
- breed classification: top-5 Oxford-IIIT Pet breeds and confidence
- pet segmentation: predicted mask and overlay
- the same resize and normalization used during training

The ZIP files from Kaggle must contain either `unet_multitask_final.pt` or `attunet_multitask_final.pt`. Set the two paths below, then run all cells.

In [ ]:
# Install only if your environment does not already have these packages.
# %pip install torch torchvision pillow matplotlib

from pathlib import Path
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

# Change these paths. In Kaggle, use paths under /kaggle/input/...
CHECKPOINT_ZIP = Path("02 Attention U net.zip")
# For the base model, use: Path("01 Base U net.zip")
INPUT_IMAGE = Path("my_pet.jpg")  # change to your image path
EXTRACT_DIR = Path("pet_checkpoints")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
print("Device:", DEVICE)

In [ ]:
# Locate the checkpoint whether you supplied a .pt file or a Kaggle ZIP.
def find_checkpoint(path):
    path = Path(path)
    if path.suffix.lower() == ".zip":
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(path) as zf:
            zf.extractall(EXTRACT_DIR)
        candidates = list(EXTRACT_DIR.rglob("*.pt"))
    elif path.suffix.lower() in {".pt", ".pth"}:
        candidates = [path]
    else:
        # Some Windows downloads unpack a PyTorch .pt archive into a folder
        # containing data.pkl, data/, byteorder, and version. Repack it.
        if path.is_dir() and (path / "data.pkl").exists() and (path / "data").is_dir():
            EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
            repacked = EXTRACT_DIR / (path.name + ".pt")
            with zipfile.ZipFile(repacked, "w", compression=zipfile.ZIP_STORED) as zf:
                for item in path.rglob("*"):
                    if item.is_file():
                        arcname = (Path(path.name) / item.relative_to(path)).as_posix()
                        zf.write(item, arcname)
            candidates = [repacked]
        else:
            candidates = list(path.rglob("*.pt")) if path.is_dir() else []
    if not candidates:
        raise FileNotFoundError("No .pt checkpoint found. Inspect the ZIP and update CHECKPOINT_ZIP.")
    preferred = [p for p in candidates if "final" in p.name.lower()]
    return preferred[0] if preferred else candidates[0]

CKPT_PATH = find_checkpoint(CHECKPOINT_ZIP)
print("Checkpoint:", CKPT_PATH)

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        h = max(ch // r, 8)
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, h), nn.ReLU(inplace=True), nn.Linear(h, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x)[..., None, None]

class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        self.se = SEBlock(cout)
    def forward(self, x): return self.se(self.net(x))

class AttentionGate(nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(f_g, f_int, 1, bias=False), nn.BatchNorm2d(f_int))
        self.W_x = nn.Sequential(nn.Conv2d(f_l, f_int, 1, bias=False), nn.BatchNorm2d(f_int))
        self.psi = nn.Sequential(nn.Conv2d(f_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, x, g): return x * self.psi(F.relu(self.W_g(g) + self.W_x(x), inplace=True))

class PetUNet(nn.Module):
    def __init__(self, n_cls, n_seg, base, dropout, attention=False):
        super().__init__(); b = base; self.attention = attention
        self.enc1, self.enc2 = DoubleConv(3,b), DoubleConv(b,b*2)
        self.enc3, self.enc4 = DoubleConv(b*2,b*4), DoubleConv(b*4,b*8)
        self.bottleneck, self.pool = DoubleConv(b*8,b*16), nn.MaxPool2d(2)
        if attention:
            self.att4, self.att3 = AttentionGate(b*8,b*8,b*4), AttentionGate(b*4,b*4,b*2)
            self.att2, self.att1 = AttentionGate(b*2,b*2,b), AttentionGate(b,b,b//2)
        self.up4, self.dec4 = nn.ConvTranspose2d(b*16,b*8,2,2), DoubleConv(b*16,b*8)
        self.up3, self.dec3 = nn.ConvTranspose2d(b*8,b*4,2,2), DoubleConv(b*8,b*4)
        self.up2, self.dec2 = nn.ConvTranspose2d(b*4,b*2,2,2), DoubleConv(b*4,b*2)
        self.up1, self.dec1 = nn.ConvTranspose2d(b*2,b,2,2), DoubleConv(b*2,b)
        self.seg_head = nn.Conv2d(b,n_seg,1); self.gap = nn.AdaptiveAvgPool2d(1); self.gmp = nn.AdaptiveMaxPool2d(1)
        self.cls_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(b*16*2+b*2,512), nn.BatchNorm1d(512), nn.ReLU(inplace=True), nn.Dropout(dropout*.5), nn.Linear(512,n_cls))
    @staticmethod
    def pad(x, skip):
        dh,dw=skip.shape[2]-x.shape[2],skip.shape[3]-x.shape[3]; return F.pad(x,[dw//2,dw-dw//2,dh//2,dh-dh//2])
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3)); bn=self.bottleneck(self.pool(e4))
        g4=self.pad(self.up4(bn),e4); s4=self.att4(e4,g4) if self.attention else e4; d=self.dec4(torch.cat([g4,s4],1))
        g3=self.pad(self.up3(d),e3); s3=self.att3(e3,g3) if self.attention else e3; d=self.dec3(torch.cat([g3,s3],1))
        g2=self.pad(self.up2(d),e2); s2=self.att2(e2,g2) if self.attention else e2; d=self.dec2(torch.cat([g2,s2],1))
        g1=self.pad(self.up1(d),e1); s1=self.att1(e1,g1) if self.attention else e1; d=self.dec1(torch.cat([g1,s1],1))
        seg=self.seg_head(d); pet_p=torch.softmax(seg,1)[:,1:2]; fg=(d*pet_p).sum((2,3))/(pet_p.sum((2,3))+1e-6)
        features=torch.cat([self.gap(bn).flatten(1),self.gmp(bn).flatten(1),F.adaptive_avg_pool2d(d,1).flatten(1),fg],1)
        return self.cls_head(features), seg

ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
# Attention U-Net final files contain metadata; Base U-Net best files are raw state_dicts.
if isinstance(ckpt, dict) and "config" in ckpt:
    config = ckpt["config"]; names = ckpt["class_names"]; state_dict = ckpt["state_dict"]
else:
    config = {"img_size": 224, "n_cls": 37, "n_seg": 2, "base_ch": 80, "cls_dropout": 0.5}
    names = ["Abyssinian", "American Bulldog", "American Pit Bull Terrier", "Basset Hound", "Beagle", "Bengal", "Birman", "Bombay", "Boxer", "British Shorthair", "Chihuahua", "Egyptian Mau", "English Cocker Spaniel", "English Setter", "German Shorthaired", "Great Pyrenees", "Havanese", "Japanese Chin", "Keeshond", "Labrador Retriever", "Maine Coon", "Miniature Pinscher", "Newfoundland", "Persian", "Pomeranian", "Pug", "Ragdoll", "Russian Blue", "Saint Bernard", "Samoyed", "Scottish Terrier", "Shiba Inu", "Siamese", "Sphynx", "Staffordshire Bull Terrier", "Wheaten Terrier", "Yorkshire Terrier"]
    state_dict = ckpt
is_attention = "att" in CKPT_PATH.name.lower() or "attention" in str(CHECKPOINT_ZIP).lower()
model = PetUNet(config["n_cls"], config["n_seg"], config["base_ch"], config.get("cls_dropout", .5), is_attention).to(DEVICE)
model.load_state_dict(state_dict); model.eval()
SIZE = config["img_size"]
print("Loaded", "Attention U-Net" if is_attention else "Base U-Net", f"| {len(names)} breeds | {SIZE}px")

In [ ]:
def make_tensor(pil):
    return TF.normalize(TF.to_tensor(pil), MEAN, STD).unsqueeze(0).to(DEVICE)

@torch.no_grad()
def predict_image(path, tta_flip=True):
    original = Image.open(path).convert("RGB"); pil = original.resize((SIZE,SIZE), Image.Resampling.BILINEAR)
    x = make_tensor(pil); cls, seg = model(x)
    if tta_flip:
        cls2, seg2 = model(torch.flip(x, [3])); cls, seg = (cls+cls2)/2, (seg+torch.flip(seg2,[3]))/2
    prob=torch.softmax(cls,1)[0]; top=torch.argsort(prob,descending=True)[:5]; mask=seg.argmax(1)[0].cpu().numpy()
    fig, ax=plt.subplots(1,3,figsize=(14,4)); ax[0].imshow(pil); ax[0].set_title("input")
    ax[1].imshow(mask,cmap="gray"); ax[1].set_title("predicted mask")
    ax[2].imshow(pil); ax[2].imshow(mask,cmap="jet",alpha=.45); ax[2].set_title("segmentation overlay")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
    print("Top breed predictions:")
    for rank,i in enumerate(top,1): print(f"{rank}. {names[int(i)]}: {float(prob[i]):.2%}")
    print(f"Predicted pet area: {mask.mean():.2%}")
    return {"breed":names[int(top[0])], "confidence":float(prob[top[0]]), "mask":mask, "top5":[(names[int(i)],float(prob[i])) for i in top]}

result = predict_image(INPUT_IMAGE)

## Usage Notes
- A 37-class checkpoint predicts breed names. A segmentation-only checkpoint cannot produce classification.
- The model was trained with augmented images, but inference uses the standard 224px resize and ImageNet normalization. Horizontal flip TTA is enabled by default.
- If loading fails, inspect the ZIP contents and set `CHECKPOINT_ZIP` to the ZIP containing the matching final `.pt` file. For a local image, set `INPUT_IMAGE` to its full path.